In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

# FLenQA probe direction through J-space

This descriptive notebook asks two connected questions: where does the FLenQA probe direction lie relative to the averaged J-Lens map, and does prompt-specific sensitivity to that direction collapse when the model changes from correct to wrong? It reuses frozen artifacts and performs no interventions.

In [ ]:
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
import jlens
import matplotlib.pyplot as plt
from datasets import load_from_disk
from tqdm.auto import tqdm
from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.evaluation import evaluate_paper_binary
from jlens_reasoning.evaluation_utils import answer_token_variants
context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
PROBE_PATH, METADATA_PATH, SPLIT_PATH = ASSET_DIR / "probes.pt", ASSET_DIR / "metadata.json", ASSET_DIR / "problem_split.json"
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
RESULT_DIR = context.runs_dir / "flenqa-probe-jlens"
JSPACE_RESULT_PATH, PROMPT_RESULT_PATH = RESULT_DIR / "jspace_summary.parquet", RESULT_DIR / "matched_sensitivity.parquet"
EXPECTED_CONTEXT_SIZES, MAX_SEQ_LEN = (2000, 3000), 4096
N_RANDOM_DIRECTIONS, BOOTSTRAP_SAMPLES = 1024, 2000

## Load frozen probes, model, and J-Lens

The saved lens.jacobians[layer] tensors are fixed, averaged J-Lens maps. Part A uses them directly. Part B uses autograd for the scalar True-minus-False margin at each hidden state and never constructs a prompt-specific Jacobian.

In [ ]:
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")); split_asset = json.loads(SPLIT_PATH.read_text(encoding="utf-8")); checkpoint = torch.load(PROBE_PATH, map_location="cpu", weights_only=False)
assert metadata["format_version"] == checkpoint["format_version"] == 1
test_ids = set(split_asset["problems"]["test"])
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, local_files_only=True).to(context.device); tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True); causal_lm.eval()
num_layers, hidden_dim = int(causal_lm.config.num_hidden_layers), int(causal_lm.config.hidden_size)
assert metadata["num_layers"] == num_layers and metadata["hidden_dim"] == hidden_dim
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
available_layers = sorted(set(lens.source_layers) & set(checkpoint["layers"]))
jacobians = {layer: lens.jacobians[layer].detach().float().cpu() for layer in available_layers}
assert all(j.shape == (hidden_dim, hidden_dim) for j in jacobians.values())
unembedding = causal_lm.get_output_embeddings().weight.detach().float().cpu()
true_ids = tuple(i for i, _ in answer_token_variants(tokenizer, ("True",))); false_ids = tuple(i for i, _ in answer_token_variants(tokenizer, ("False",)))
answer_direction = unembedding[list(true_ids)].mean(0) - unembedding[list(false_ids)].mean(0); answer_direction = answer_direction / answer_direction.norm()
dataset = load_from_disk(context.datasets_dir / "flenqa"); raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset; rows = normalize_rows(raw_rows, full=True)
test_prompts = [p for p in prepare_prompts([r for r in rows if r.problem_id in test_ids]) if {x.ctx_size for x in p.provenance}.pop() in EXPECTED_CONTEXT_SIZES]
prompt_context_sizes = {p.prompt_id: {x.ctx_size for x in p.provenance}.pop() for p in test_prompts}; model_records = {r["prompt_id"]: r for r in pq.read_table(MODEL_OUTPUT_PATH).to_pylist()}
examples = []
for prompt in test_prompts:
    evaluation = evaluate_paper_binary(model_records[prompt.prompt_id]["generated_text"], expected=prompt.label)
    examples.append({"prompt_id":prompt.prompt_id, "problem_id":prompt.problem_id, "ctx_size":prompt_context_sizes[prompt.prompt_id], "task":prompt.task, "match_key":(prompt.problem_id, prompt.task, prompt.provenance[0].padding_type, prompt.provenance[0].dispersion), "label":int(prompt.label), "model_correct":bool(evaluation.correct), "prompt":prompt})
assert {e["ctx_size"] for e in examples} == set(EXPECTED_CONTEXT_SIZES)

## Part A — probe direction in averaged J-space

For each layer, d = w / ||w|| and J_mean is the saved averaged map. Measure ||J_mean d|| and cosine alignment with the True-vs-False output direction. Random unit directions in the same hidden space provide a percentile and z-score for both quantities.

In [ ]:
def unit(v): return v / torch.linalg.vector_norm(v, dim=-1, keepdim=True)
def percentile_z(value, null):
    null = np.asarray(null); return float((null <= value).mean() * 100), float((value - null.mean()) / (null.std(ddof=1) + 1e-12))
rng = torch.Generator().manual_seed(0); random_directions = unit(torch.randn(N_RANDOM_DIRECTIONS, hidden_dim, generator=rng)); jspace_rows = []
for layer in available_layers:
    d = unit(checkpoint["layers"][layer]["unit_weight"].float()); propagated = jacobians[layer].mv(d); norm = float(propagated.norm()); alignment = float(torch.dot(unit(propagated), answer_direction))
    null_propagated = random_directions @ jacobians[layer].T; null_norms = torch.linalg.vector_norm(null_propagated, dim=1).numpy(); null_alignments = (null_propagated @ answer_direction).numpy() / (null_norms + 1e-12)
    norm_pct, norm_z = percentile_z(norm, null_norms); align_pct, align_z = percentile_z(alignment, null_alignments)
    jspace_rows.append({"layer":layer, "propagation_norm":norm, "answer_alignment":alignment, "norm_percentile":norm_pct, "norm_z":norm_z, "alignment_percentile":align_pct, "alignment_z":align_z})
jspace_summary = pd.DataFrame(jspace_rows); RESULT_DIR.mkdir(parents=True, exist_ok=True); pq.write_table(pa.Table.from_pandas(jspace_summary, preserve_index=False), JSPACE_RESULT_PATH); display(jspace_summary.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4)); axes[0].plot(jspace_summary.layer, jspace_summary.propagation_norm, marker="o"); axes[0].set(title="Propagation ||J_mean d||", ylabel="norm"); axes[1].plot(jspace_summary.layer, jspace_summary.answer_alignment, marker="o"); axes[1].axhline(0, color="black", lw=1); axes[1].set(title="Alignment with True-False direction", ylabel="cosine")
for ax in axes: ax.set_xlabel("layer"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## Part B — prompt-specific downstream sensitivity

For each prompt and layer, m = logit(True) − logit(False). The probe margin is oriented toward gold, and d is the corresponding gold-oriented unit probe direction. Autograd computes sensitivity = grad_h(m) · d at the final analyzed position.

In [ ]:
def prompt_sensitivities(item):
    encoded = tokenizer(item["prompt"].text, return_tensors="pt", truncation=False); input_ids = encoded["input_ids"].to(context.device); assert 0 < input_ids.shape[1] <= MAX_SEQ_LEN
    outputs = causal_lm(input_ids=input_ids, output_hidden_states=True, use_cache=False); hidden_states = outputs.hidden_states; logits = outputs.logits[0, -1].float(); margin = logits[list(true_ids)].mean() - logits[list(false_ids)].mean()
    grads = torch.autograd.grad(margin, tuple(hidden_states[layer + 1] for layer in available_layers)); gold_sign = 1.0 if item["label"] == 1 else -1.0; rows = []
    for layer, grad in zip(available_layers, grads):
        asset = checkpoint["layers"][layer]; hidden = hidden_states[layer + 1][0, -1].detach().float().cpu(); raw = torch.dot(hidden - asset["training_mean"].float(), asset["weight"].float()) + asset["bias"].float(); d = gold_sign * unit(asset["unit_weight"].float()); sensitivity = torch.dot(grad[0, -1].float().cpu(), d)
        rows.append({"prompt_id":item["prompt_id"], "problem_id":item["problem_id"], "ctx_size":item["ctx_size"], "match_key":item["match_key"], "label":item["label"], "model_correct":item["model_correct"], "layer":layer, "output_margin":float(margin.detach().cpu()), "probe_margin":float((gold_sign * raw).cpu()), "probe_correct":bool((gold_sign * raw) > 0), "sensitivity":float(sensitivity.cpu())})
    return rows
prompt_rows = []
for item in tqdm(examples, desc="Computing prompt sensitivities"): prompt_rows.extend(prompt_sensitivities(item))
prompt_results = pd.DataFrame(prompt_rows); assert len(prompt_results) == len(examples) * len(available_layers)
wide = prompt_results.pivot(index=["match_key", "problem_id", "layer"], columns="ctx_size").dropna(subset=[("model_correct",2000),("model_correct",3000)]); pair_rows = []
for (match_key, problem_id, layer), row in wide.iterrows():
    probe_ok = bool(row[("probe_correct",2000)] and row[("probe_correct",3000)]); c2000, c3000 = bool(row[("model_correct",2000)]), bool(row[("model_correct",3000)])
    pair_type = "failure-transition" if c2000 and not c3000 and probe_ok else "stable-control" if c2000 and c3000 and probe_ok else None
    if pair_type: pair_rows.append({"match_key":match_key, "problem_id":problem_id, "layer":layer, "pair_type":pair_type, "prompt_id_2000":row[("prompt_id",2000)], "prompt_id_3000":row[("prompt_id",3000)], "probe_margin_2000":row[("probe_margin",2000)], "probe_margin_3000":row[("probe_margin",3000)], "probe_correct_2000":row[("probe_correct",2000)], "probe_correct_3000":row[("probe_correct",3000)], "model_correct_2000":c2000, "model_correct_3000":c3000, "sensitivity_2000":row[("sensitivity",2000)], "sensitivity_3000":row[("sensitivity",3000)], "delta_sensitivity":row[("sensitivity",3000)]-row[("sensitivity",2000)]})
matched_results = pd.DataFrame(pair_rows); pq.write_table(pa.Table.from_pandas(matched_results, preserve_index=False), PROMPT_RESULT_PATH); display(matched_results.head()); print(f"Saved {len(prompt_results):,} prompt-layer rows and {len(matched_results):,} matched rows")

## Matched analysis

Keep matched problems where the probe is correct at both lengths. Compare delta_sensitivity = sensitivity_3000 − sensitivity_2000 between failure-transition pairs and stable controls. The interval is a simple bootstrap 95% CI for the difference in group means.

In [ ]:
def bootstrap_difference(a, b, samples=BOOTSTRAP_SAMPLES, seed=0):
    rng = np.random.default_rng(seed); a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) == 0 or len(b) == 0: return np.nan, np.nan, np.nan
    draws = rng.choice(a, (samples,len(a)), replace=True).mean(1) - rng.choice(b, (samples,len(b)), replace=True).mean(1)
    return float(a.mean()-b.mean()), float(np.quantile(draws,.025)), float(np.quantile(draws,.975))
stats = []
for layer, group in matched_results.groupby("layer"):
    a = group.loc[group.pair_type == "failure-transition", "delta_sensitivity"]; b = group.loc[group.pair_type == "stable-control", "delta_sensitivity"]; difference, low, high = bootstrap_difference(a,b,seed=layer); stats.append({"layer":layer,"difference":difference,"ci_low":low,"ci_high":high,"n_transition":len(a),"n_stable":len(b)})
sensitivity_summary = pd.DataFrame(stats); display(sensitivity_summary.round(3))
fig, ax = plt.subplots(figsize=(8,4)); ax.axhline(0,color="black",lw=1); ax.plot(sensitivity_summary.layer,sensitivity_summary.difference,marker="o"); ax.fill_between(sensitivity_summary.layer,sensitivity_summary.ci_low,sensitivity_summary.ci_high,alpha=.2); ax.set(title="Sensitivity change: failure-transition - stable",xlabel="layer",ylabel="delta sensitivity (95% bootstrap CI)"); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

In [ ]:
# Keep names used by earlier reports while retaining the new compact summaries.
jspace_summary["answer_effect"] = jspace_summary["answer_alignment"]
prompt_results["gold_probe_score"] = prompt_results["probe_margin"]
prompt_results.groupby(["layer", "model_correct"]).size()